In [ ]:
# ================================================================
# Notebook 03 — Federated Learning Results Visualization
# ================================================================
# Run this notebook after completing all experiments to generate:
#   - AUPRC trajectory plots per bank across rounds
#   - F1-Score trajectory plots
#   - Comparative bar charts (Local vs Federated vs Centralized)
#   - Precision-Recall curves for each model condition
#   - Feature importance chart for federated global model
# ================================================================

1: Imports and Configuration

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import numpy as np
import pandas as pd
import seaborn as sns
import xgboost as xgb
from sklearn.metrics import precision_recall_curve, auc

# Plot styling
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.2)
plt.rcParams.update({
    "figure.dpi": 150,
    "axes.titlesize": 14,
    "axes.labelsize": 12,
    "figure.figsize": (10, 6),
})

RESULTS_DIR  = Path("experiments/results")
MODELS_DIR   = Path("models")
FIGURES_DIR  = Path("notebooks/figures")
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

BANK_IDS     = ["bank1", "bank2", "bank3"]
BANK_LABELS  = {
    "bank1": "Bank 1 (High-Risk)",
    "bank2": "Bank 2 (Retail/Blind Spot)",
    "bank3": "Bank 3 (Mixed)",
}
BANK_COLORS  = {
    "bank1": "#2196F3",   # Blue
    "bank2": "#F44336",   # Red  — highlighted as blind spot
    "bank3": "#4CAF50",   # Green
}
NUM_ROUNDS   = 5

print("Configuration loaded. ✓")

2: Load All Results

In [ ]:
def load_trajectory(bank_id: str) -> pd.DataFrame:
    """Load the full per-round trajectory for a single bank."""
    path = (
        RESULTS_DIR
        / "federated"
        / f"{bank_id}_full_trajectory.json"
    )
    with open(path, "r") as f:
        data = json.load(f)
    return pd.DataFrame(data["trajectory"])


def load_centralized_results() -> dict:
    """Load centralized baseline results."""
    path = (
        RESULTS_DIR
        / "central_baseline"
        / "centralized_baseline_results.json"
    )
    with open(path, "r") as f:
        return json.load(f)


# Load trajectories
trajectories = {
    bank_id: load_trajectory(bank_id)
    for bank_id in BANK_IDS
}

central_results = load_centralized_results()

print("All results loaded. ✓")
for bank_id, df in trajectories.items():
    print(f"  {bank_id}: {len(df)} rounds")


3: AUPRC Trajectory Plot

In [ ]:
fig, ax = plt.subplots(figsize=(11, 6))

for bank_id in BANK_IDS:
    df = trajectories[bank_id]
    ax.plot(
        df["round"],
        df["auprc"],
        marker="o",
        linewidth=2.5,
        markersize=8,
        color=BANK_COLORS[bank_id],
        label=BANK_LABELS[bank_id],
        zorder=3
    )

    # Annotate final round value
    final = df.iloc[-1]
    ax.annotate(
        f"{final['auprc']:.4f}",
        xy=(final["round"], final["auprc"]),
        xytext=(8, 0),
        textcoords="offset points",
        fontsize=9,
        color=BANK_COLORS[bank_id]
    )

# Centralized baseline reference line
ax.axhline(
    y=central_results["auprc"],
    color="black",
    linestyle="--",
    linewidth=1.5,
    alpha=0.7,
    label=f"Centralized Baseline (AUPRC = {central_results['auprc']:.4f})"
)

ax.set_xlabel("Federation Round", fontsize=12)
ax.set_ylabel("AUPRC", fontsize=12)
ax.set_title(
    "AUPRC Trajectory Across Federated Communication Rounds",
    fontsize=14, fontweight="bold"
)
ax.set_xticks(range(0, NUM_ROUNDS + 1))
ax.set_ylim(-0.05, 1.05)
ax.legend(loc="lower right", fontsize=10)
ax.grid(True, alpha=0.4)

plt.tight_layout()
plt.savefig(
    FIGURES_DIR / "auprc_trajectory.png",
    dpi=150, bbox_inches="tight"
)
plt.show()
print("Figure saved → notebooks/figures/auprc_trajectory.png")

4: F1-Score Trajectory Plot

In [ ]:
fig, ax = plt.subplots(figsize=(11, 6))

for bank_id in BANK_IDS:
    df = trajectories[bank_id]
    ax.plot(
        df["round"],
        df["f1_score"],
        marker="s",
        linewidth=2.5,
        markersize=8,
        color=BANK_COLORS[bank_id],
        label=BANK_LABELS[bank_id],
        zorder=3
    )

    final = df.iloc[-1]
    ax.annotate(
        f"{final['f1_score']:.4f}",
        xy=(final["round"], final["f1_score"]),
        xytext=(8, 0),
        textcoords="offset points",
        fontsize=9,
        color=BANK_COLORS[bank_id]
    )

ax.axhline(
    y=central_results["f1_score"],
    color="black",
    linestyle="--",
    linewidth=1.5,
    alpha=0.7,
    label=f"Centralized Baseline (F1 = {central_results['f1_score']:.4f})"
)

ax.set_xlabel("Federation Round", fontsize=12)
ax.set_ylabel("F1-Score", fontsize=12)
ax.set_title(
    "F1-Score Trajectory Across Federated Communication Rounds",
    fontsize=14, fontweight="bold"
)
ax.set_xticks(range(0, NUM_ROUNDS + 1))
ax.set_ylim(-0.05, 1.05)
ax.legend(loc="lower right", fontsize=10)
ax.grid(True, alpha=0.4)

plt.tight_layout()
plt.savefig(
    FIGURES_DIR / "f1_trajectory.png",
    dpi=150, bbox_inches="tight"
)
plt.show()
print("Figure saved → notebooks/figures/f1_trajectory.png")

5: Comparative Bar Chart

AUPRC: Local-Only vs Federated Round 5 vs Centralized

In [ ]:

conditions = ["Local-Only", "FL Round 5", "Centralized"]
x          = np.arange(len(BANK_IDS))
bar_width  = 0.25

fig, ax = plt.subplots(figsize=(12, 7))

# Gather values per condition
local_auprc = [
    trajectories[b].iloc[0]["auprc"]
    for b in BANK_IDS
]
fed_auprc = [
    trajectories[b].iloc[-1]["auprc"]
    for b in BANK_IDS
]
central_auprc = [central_results["auprc"]] * 3

bar_colors   = ["#90CAF9", "#1565C0", "#424242"]
bar_patterns = ["", "//", ".."]

for i, (vals, label, color, hatch) in enumerate(zip(
    [local_auprc, fed_auprc, central_auprc],
    conditions,
    bar_colors,
    bar_patterns
)):
    bars = ax.bar(
        x + i * bar_width,
        vals,
        bar_width,
        label=label,
        color=color,
        hatch=hatch,
        edgecolor="white",
        linewidth=0.8,
        zorder=3
    )
    # Value labels on bars
    for bar, val in zip(bars, vals):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.01,
            f"{val:.4f}",
            ha="center",
            va="bottom",
            fontsize=8.5,
            fontweight="bold"
        )

ax.set_xlabel("Client Bank", fontsize=12)
ax.set_ylabel("AUPRC", fontsize=12)
ax.set_title(
    "AUPRC Comparison: Local-Only vs Federated vs Centralized",
    fontsize=14, fontweight="bold"
)
ax.set_xticks(x + bar_width)
ax.set_xticklabels(
    [BANK_LABELS[b] for b in BANK_IDS],
    fontsize=10
)
ax.set_ylim(0, 1.12)
ax.legend(fontsize=10)
ax.grid(axis="y", alpha=0.4)

# Highlight Bank 2 blind spot recovery
ax.annotate(
    "Blind Spot\nResolved ↑",
    xy=(1 + bar_width, fed_auprc[1] + 0.02),
    xytext=(1 + bar_width + 0.15, fed_auprc[1] + 0.15),
    fontsize=9,
    color="#C62828",
    fontweight="bold",
    arrowprops=dict(
        arrowstyle="->",
        color="#C62828",
        lw=1.5
    )
)

plt.tight_layout()
plt.savefig(
    FIGURES_DIR / "auprc_comparison_bar.png",
    dpi=150, bbox_inches="tight"
)
plt.show()
print("Figure saved → notebooks/figures/auprc_comparison_bar.png")


6: Precision-Recall Curves

In [ ]:
def plot_pr_curve(
    model_path: str,
    test_df: pd.DataFrame,
    label: str,
    color: str,
    ax,
    round_num: int = 0
):
    """Plot a single Precision-Recall curve on the given axes."""
    model = xgb.XGBClassifier()
    model.load_model(model_path)

    X_test = test_df.drop(columns=["isFraud"])
    y_test = test_df["isFraud"]

    y_prob = model.predict_proba(X_test)[:, 1]
    prec, rec, _ = precision_recall_curve(y_test, y_prob)
    auprc_val    = auc(rec, prec)

    ax.plot(
        rec, prec,
        color=color,
        linewidth=2.0,
        label=f"{label} (AUPRC = {auprc_val:.4f})"
    )
    return auprc_val


# Load global test set
global_test_df = pd.read_csv("data/processed/global_test.csv")

fig, ax = plt.subplots(figsize=(10, 7))

# Local-Only models
local_model_configs = [
    ("bank1", BANK_COLORS["bank1"], "Bank 1 Local-Only"),
    ("bank2", BANK_COLORS["bank2"], "Bank 2 Local-Only"),
    ("bank3", BANK_COLORS["bank3"], "Bank 3 Local-Only"),
]

for bank_id, color, label in local_model_configs:
    model_path = str(
        MODELS_DIR / "local" / f"{bank_id}_local_model.json"
    )
    if Path(model_path).exists():
        plot_pr_curve(
            model_path, global_test_df,
            label, color, ax
        )

# Centralized model
central_path = str(
    MODELS_DIR / "local" / "centralized_model.json"
)
if Path(central_path).exists():
    plot_pr_curve(
        central_path, global_test_df,
        "Centralized (Privacy-Violated)", "black", ax
    )

# Federated model — Round 5
fed_path = str(
    MODELS_DIR / "global" / f"global_model_round_{NUM_ROUNDS}.json"
)
if Path(fed_path).exists():
    plot_pr_curve(
        fed_path, global_test_df,
        f"Federated Global (Round {NUM_ROUNDS})",
        "#9C27B0", ax
    )

# Random classifier baseline
ax.axhline(
    y=global_test_df["isFraud"].mean(),
    color="gray",
    linestyle=":",
    linewidth=1.5,
    label=f"Random Classifier "
          f"(Prevalence = {global_test_df['isFraud'].mean():.4f})"
)

ax.set_xlabel("Recall", fontsize=12)
ax.set_ylabel("Precision", fontsize=12)
ax.set_title(
    "Precision-Recall Curves — All Models",
    fontsize=14, fontweight="bold"
)
ax.set_xlim(0, 1.02)
ax.set_ylim(0, 1.05)
ax.legend(loc="upper right", fontsize=9)
ax.grid(True, alpha=0.4)

plt.tight_layout()
plt.savefig(
    FIGURES_DIR / "precision_recall_curves.png",
    dpi=150, bbox_inches="tight"
)
plt.show()
print("Figure saved → notebooks/figures/precision_recall_curves.png")

7: Feature Importance Chart

In [ ]:
def get_feature_importance(
    model_path: str,
    feature_names: list,
    importance_type: str = "gain"
) -> pd.Series:
    """Extract feature importance scores from a saved model."""
    model = xgb.XGBClassifier()
    model.load_model(model_path)
    scores = model.get_booster().get_score(
        importance_type=importance_type
    )
    series = pd.Series(scores).reindex(feature_names, fill_value=0)
    return series.sort_values(ascending=False)


# Load feature names from global test set
feature_names = [
    c for c in global_test_df.columns
    if c != "isFraud"
]

fed_model_path = str(
    MODELS_DIR / "global" / f"global_model_round_{NUM_ROUNDS}.json"
)

if Path(fed_model_path).exists():
    importance = get_feature_importance(
        fed_model_path,
        feature_names,
        importance_type="gain"
    )

    # Plot top 12 features
    top_n   = 12
    top_imp = importance.head(top_n)

    fig, ax = plt.subplots(figsize=(10, 7))

    colors = [
        "#1565C0" if i < 2 else "#42A5F5"
        for i in range(len(top_imp))
    ]

    bars = ax.barh(
        top_imp.index[::-1],
        top_imp.values[::-1],
        color=colors[::-1],
        edgecolor="white",
        linewidth=0.5,
        zorder=3
    )

    # Value labels
    for bar, val in zip(bars, top_imp.values[::-1]):
        ax.text(
            bar.get_width() + 0.01 * top_imp.max(),
            bar.get_y() + bar.get_height() / 2,
            f"{val:.1f}",
            va="center",
            fontsize=9
        )

    ax.set_xlabel("Feature Importance (Gain)", fontsize=12)
    ax.set_title(
        f"Top {top_n} Feature Importance Scores\n"
        f"Federated Global Model — Round {NUM_ROUNDS}",
        fontsize=14, fontweight="bold"
    )
    ax.grid(axis="x", alpha=0.4)
    ax.set_xlim(0, top_imp.max() * 1.15)

    plt.tight_layout()
    plt.savefig(
        FIGURES_DIR / "feature_importance.png",
        dpi=150, bbox_inches="tight"
    )
    plt.show()
    print("Figure saved → notebooks/figures/feature_importance.png")


8: Bank 2 Recovery Summary

In [ ]:
# Focused visualization of the blind spot problem resolution

bank2_df = trajectories["bank2"]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

metrics   = ["auprc",    "f1_score"]
ylabels   = ["AUPRC",    "F1-Score"]
ref_vals  = [
    central_results["auprc"],
    central_results["f1_score"]
]

for ax, metric, ylabel, ref_val in zip(
    axes, metrics, ylabels, ref_vals
):
    ax.plot(
        bank2_df["round"],
        bank2_df[metric],
        marker="o",
        linewidth=2.5,
        markersize=10,
        color=BANK_COLORS["bank2"],
        label="Bank 2 (Federated)",
        zorder=3
    )

    # Annotate each point
    for _, row in bank2_df.iterrows():
        ax.annotate(
            f"{row[metric]:.4f}",
            xy=(row["round"], row[metric]),
            xytext=(0, 10),
            textcoords="offset points",
            ha="center",
            fontsize=9,
            color=BANK_COLORS["bank2"]
        )

    # Centralized reference line
    ax.axhline(
        y=ref_val,
        color="black",
        linestyle="--",
        linewidth=1.5,
        alpha=0.7,
        label=f"Centralized Baseline ({ref_val:.4f})"
    )

    # Shade the recovery region
    ax.fill_between(
        bank2_df["round"],
        bank2_df[metric],
        0,
        alpha=0.12,
        color=BANK_COLORS["bank2"]
    )

    ax.set_xlabel("Federation Round", fontsize=11)
    ax.set_ylabel(ylabel, fontsize=11)
    ax.set_title(
        f"Bank 2 — {ylabel} Recovery\n"
        f"(Blind Spot → Functional Detector)",
        fontsize=12, fontweight="bold"
    )
    ax.set_xticks(range(0, NUM_ROUNDS + 1))
    ax.set_ylim(-0.05, 1.05)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.4)

    # Annotate Round 0 as blind spot
    ax.annotate(
        "Total Failure\n(Blind Spot)",
        xy=(0, 0),
        xytext=(0.3, 0.08),
        fontsize=8.5,
        color="#B71C1C",
        fontweight="bold",
        arrowprops=dict(
            arrowstyle="->",
            color="#B71C1C",
            lw=1.2
        )
    )

plt.suptitle(
    "Bank 2 Blind Spot Resolution via Federated Learning",
    fontsize=14, fontweight="bold", y=1.02
)
plt.tight_layout()
plt.savefig(
    FIGURES_DIR / "bank2_recovery.png",
    dpi=150, bbox_inches="tight"
)
plt.show()
print("Figure saved → notebooks/figures/bank2_recovery.png")

9: Full Results Table

In [ ]:
print("\n" + "═" * 75)
print("  COMPLETE RESULTS TABLE")
print("═" * 75)

rows = []
for bank_id in BANK_IDS:
    for _, row in trajectories[bank_id].iterrows():
        rows.append({
            "Bank":      BANK_LABELS[bank_id],
            "Round":     int(row["round"]),
            "AUPRC":     f"{row['auprc']:.4f}",
            "F1-Score":  f"{row['f1_score']:.4f}",
            "Precision": f"{row['precision']:.4f}",
            "Recall":    f"{row['recall']:.4f}",
            "Condition": (
                "Local-Only" if row["round"] == 0
                else f"Federated"
            )
        })

results_table = pd.DataFrame(rows)
print(results_table.to_string(index=False))

# Save as CSV for thesis tables
results_table.to_csv(
    FIGURES_DIR / "full_results_table.csv",
    index=False
)

# Centralized row
print(
    f"\n  {'Centralized':<40} — "
    f"AUPRC: {central_results['auprc']:.4f} "
    f"| F1: {central_results['f1_score']:.4f} "
    f"[Privacy-Violated Upper Bound]"
)
print(
    "\n  [NOTE] Accuracy is excluded from all results.\n"
    "         Reason: A classifier predicting all-legitimate\n"
    "         achieves 99.87% accuracy under 0.13% fraud prevalence.\n"
)

print("Full results table saved → notebooks/figures/full_results_table.csv")